# STAT3612 Group Project: Multimodal Presurgical Brain Tumor Classification

This notebook implements a complete pipeline for brain tumor classification using multimodal data:
- **Image Features**: 2048-d ResNet-50 features from 4 MRI modalities
- **Radiomics Features**: 5 radiomic features per modality
- **Clinical Information**: Sex, Age, Tumor Location, Signal Intensities
- **Radiology Reports**: Text features via TF-IDF

**Target**: 5-class classification (Glioma, Meningioma, Brain Metastase Tumour, Tumors of the sellar region, Pineal tumour and Choroid plexus tumour)

**Evaluation**: Macro F1 Score (Kaggle metric)

## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import lightgbm as lgb

from collections import Counter

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Base directory - adjust if needed
BASE_DIR = '.'  # Current directory where the data files are
print('Pipeline ready.')

ModuleNotFoundError: No module named 'xgboost'

## 2. Data Loading

In [ ]:
# Load JSON metadata
with open(os.path.join(BASE_DIR, 'train.json')) as f:
    train_meta = json.load(f)
with open(os.path.join(BASE_DIR, 'val.json')) as f:
    val_meta = json.load(f)
with open(os.path.join(BASE_DIR, 'test.json')) as f:
    test_meta = json.load(f)

print(f'Train: {len(train_meta)} cases')
print(f'Val:   {len(val_meta)} cases')
print(f'Test:  {len(test_meta)} cases')

# Label distribution
train_labels = {k: v['Overall_class'] for k, v in train_meta.items()}
val_labels = {k: v['Overall_class'] for k, v in val_meta.items()}

print('\nTraining label distribution:')
for cls, cnt in sorted(Counter(train_labels.values()).items()):
    print(f'  {cls}: {cnt}')

## 3. Feature Engineering

### 3.1 Image Features (ResNet-50, 2048-d per modality)

In [ ]:
MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']

def load_image_features(meta_dict, base_dir=BASE_DIR):
    """Load and concatenate image features for all cases.
    For each case, concatenate features from all 4 modalities (zero-pad if missing).
    Result: (n_cases, 4*2048) = (n_cases, 8192)
    """
    case_ids = sorted(meta_dict.keys(), key=int)
    features = []
    
    for cid in case_ids:
        info = meta_dict[cid]
        case_feats = []
        
        for mod in MODALITIES:
            npy_path = os.path.join(base_dir, 'image_features', 'image_features', cid, mod, 'image.npy')
            if os.path.exists(npy_path):
                arr = np.load(npy_path)
                case_feats.append(arr)
            else:
                case_feats.append(np.zeros(2048, dtype=np.float32))
        
        features.append(np.concatenate(case_feats))
    
    return np.array(features), case_ids

print('Loading image features...')
X_img_train, train_ids = load_image_features(train_meta)
X_img_val, val_ids = load_image_features(val_meta)
X_img_test, test_ids = load_image_features(test_meta)

print(f'Image features - Train: {X_img_train.shape}, Val: {X_img_val.shape}, Test: {X_img_test.shape}')

### 3.2 Radiomics Features

In [ ]:
RADIOMICS_FEAT_COLS = ['rad_firstorder_Mean', 'rad_firstorder_Entropy', 
                       'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 
                       'rad_glcm_JointEntropy']

def load_radiomics(split, case_ids, base_dir=BASE_DIR):
    """Load radiomics features for all 4 modalities, merge by case_id.
    Result: (n_cases, 4*5) = (n_cases, 20)
    """
    all_feats = pd.DataFrame({'case_id': [int(c) for c in case_ids]})
    
    for mod in MODALITIES:
        fname = os.path.join(base_dir, 'radiomics_info', split, f'{mod}_radiomics_{split}.csv')
        df = pd.read_csv(fname)
        # Rename feature columns with modality prefix
        rename_dict = {col: f'{mod}_{col}' for col in RADIOMICS_FEAT_COLS}
        df = df[['case_id'] + RADIOMICS_FEAT_COLS].rename(columns=rename_dict)
        all_feats = all_feats.merge(df, on='case_id', how='left')
    
    # Fill NaN with 0 (missing modality)
    feat_cols = [c for c in all_feats.columns if c != 'case_id']
    all_feats[feat_cols] = all_feats[feat_cols].fillna(0)
    
    # Ensure order matches case_ids
    all_feats = all_feats.set_index('case_id').loc[[int(c) for c in case_ids]]
    return all_feats.values

X_rad_train = load_radiomics('train', train_ids)
X_rad_val = load_radiomics('val', val_ids)
X_rad_test = load_radiomics('test', test_ids)

print(f'Radiomics features - Train: {X_rad_train.shape}, Val: {X_rad_val.shape}, Test: {X_rad_test.shape}')

### 3.3 Clinical Features

In [ ]:
def load_clinical(split, case_ids, base_dir=BASE_DIR):
    """Load and encode clinical features: Sex, Age, Tumor Location, Signal Intensities."""
    fname = os.path.join(base_dir, 'clinical_information', f'{split}_patient_info.csv')
    df = pd.read_csv(fname)
    df = df.set_index('case_id').loc[[int(c) for c in case_ids]].reset_index()
    return df

df_clin_train = load_clinical('train', train_ids)
df_clin_val = load_clinical('val', val_ids)
df_clin_test = load_clinical('test', test_ids)

# Encode Sex
sex_map = {'female': 0, 'male': 1, 'unknown': 2}
for df in [df_clin_train, df_clin_val, df_clin_test]:
    df['Sex_enc'] = df['Sex'].map(sex_map)

# Age: fill NaN with median from training
train_age_median = df_clin_train['Age'].median()
print(f'Training Age median (non-null): {train_age_median}')
for df in [df_clin_train, df_clin_val, df_clin_test]:
    df['Age_filled'] = df['Age'].fillna(train_age_median)

# Signal Intensity encoding: ordinal
intensity_map = {'hypointense': 0, 'isointense': 1, 'hyperintense': 2, 'heterogeneous': 3, 'homogeneous': 4, 'unknown': 5}
intensity_cols = ['Signal Intensity (T1)', 'Signal Intensity (T1c)', 
                  'Signal Intensity (T2)', 'Signal Intensity (T2-FLAIR)']
for df in [df_clin_train, df_clin_val, df_clin_test]:
    for col in intensity_cols:
        df[col + '_enc'] = df[col].map(intensity_map).fillna(5).astype(int)

# Tumor Location: extract key anatomical terms
location_keywords = ['frontal', 'temporal', 'parietal', 'occipital', 'cerebellum',
                     'sellar', 'sella', 'pituitary', 'pineal', 'ventricle',
                     'brainstem', 'cerebellopontine', 'thalamus', 'basal',
                     'left', 'right', 'bilateral', 'midline']

for df in [df_clin_train, df_clin_val, df_clin_test]:
    loc_lower = df['Tumor Location'].str.lower().fillna('')
    for kw in location_keywords:
        df[f'loc_{kw}'] = loc_lower.str.contains(kw).astype(int)

# Assemble clinical feature matrix
clinical_feat_cols = (['Sex_enc', 'Age_filled'] + 
                      [col + '_enc' for col in intensity_cols] +
                      [f'loc_{kw}' for kw in location_keywords])

X_clin_train = df_clin_train[clinical_feat_cols].values.astype(float)
X_clin_val = df_clin_val[clinical_feat_cols].values.astype(float)
X_clin_test = df_clin_test[clinical_feat_cols].values.astype(float)

print(f'Clinical features - Train: {X_clin_train.shape}, Val: {X_clin_val.shape}, Test: {X_clin_test.shape}')

### 3.4 Text Features (Radiology Reports via TF-IDF)

In [ ]:
def get_reports(meta_dict, case_ids):
    """Extract radiology reports in order."""
    reports = []
    for cid in case_ids:
        report = meta_dict[cid].get('report', '')
        if report is None:
            report = ''
        reports.append(report)
    return reports

reports_train = get_reports(train_meta, train_ids)
reports_val = get_reports(val_meta, val_ids)
reports_test = get_reports(test_meta, test_ids)

# TF-IDF with bigrams, limited to top 500 features
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2), stop_words='english', min_df=3)
X_text_train = tfidf.fit_transform(reports_train).toarray()
X_text_val = tfidf.transform(reports_val).toarray()
X_text_test = tfidf.transform(reports_test).toarray()

print(f'Text features - Train: {X_text_train.shape}, Val: {X_text_val.shape}, Test: {X_text_test.shape}')
print(f'Top TF-IDF terms: {tfidf.get_feature_names_out()[:20]}')

### 3.5 Combine All Features & Standardize

In [ ]:
# PCA on image features to reduce dimensionality (8192 -> 256)
pca_img = PCA(n_components=256, random_state=SEED)
X_img_train_pca = pca_img.fit_transform(X_img_train)
X_img_val_pca = pca_img.transform(X_img_val)
X_img_test_pca = pca_img.transform(X_img_test)
print(f'PCA explained variance (256 components): {pca_img.explained_variance_ratio_.sum():.4f}')

# Combine all features
X_train_all = np.hstack([X_img_train_pca, X_rad_train, X_clin_train, X_text_train])
X_val_all = np.hstack([X_img_val_pca, X_rad_val, X_clin_val, X_text_val])
X_test_all = np.hstack([X_img_test_pca, X_rad_test, X_clin_test, X_text_test])

print(f'\nCombined features: {X_train_all.shape[1]} dimensions')
print(f'  - Image PCA: 256')
print(f'  - Radiomics: {X_rad_train.shape[1]}')
print(f'  - Clinical:  {X_clin_train.shape[1]}')
print(f'  - Text:      {X_text_train.shape[1]}')

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_all)
X_val_scaled = scaler.transform(X_val_all)
X_test_scaled = scaler.transform(X_test_all)

# Prepare labels
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform([train_labels[cid] for cid in train_ids])
y_val = label_encoder.transform([val_labels[cid] for cid in val_ids])

print(f'\nLabel encoding: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}')
print(f'y_train shape: {y_train.shape}, y_val shape: {y_val.shape}')

## 4. Class Imbalance Handling

In [ ]:
# Compute class weights for handling imbalanced classes
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))
print('Class weights (balanced):')
for cls_idx, weight in class_weight_dict.items():
    cls_name = label_encoder.inverse_transform([cls_idx])[0]
    print(f'  {cls_name}: {weight:.3f}')

# Also compute sample weights for XGBoost/LightGBM
sample_weights_train = np.array([class_weight_dict[y] for y in y_train])

## 5. Model Training & Evaluation

### 5.1 Evaluation Helper

In [ ]:
def evaluate_model(model, X_val, y_val, model_name='Model'):
    """Evaluate a model and print detailed metrics."""
    y_pred = model.predict(X_val)
    
    f1_macro = f1_score(y_val, y_pred, average='macro')
    f1_weighted = f1_score(y_val, y_pred, average='weighted')
    
    print(f'\n{"="*60}')
    print(f'{model_name}')
    print(f'{"="*60}')
    print(f'Macro F1:    {f1_macro:.4f}')
    print(f'Weighted F1: {f1_weighted:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))
    
    return f1_macro, y_pred

results = {}  # Store results for comparison

### 5.2 Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', 
                        multi_class='multinomial', solver='lbfgs', random_state=SEED)
lr.fit(X_train_scaled, y_train)
f1, _ = evaluate_model(lr, X_val_scaled, y_val, 'Logistic Regression')
results['Logistic Regression'] = f1

### 5.3 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_leaf=2,
                            class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
f1, _ = evaluate_model(rf, X_val_scaled, y_val, 'Random Forest')
results['Random Forest'] = f1

### 5.4 SVM (RBF Kernel)

In [ ]:
svm = SVC(C=10, kernel='rbf', gamma='scale', class_weight='balanced', 
          random_state=SEED, probability=True)
svm.fit(X_train_scaled, y_train)
f1, _ = evaluate_model(svm, X_val_scaled, y_val, 'SVM (RBF)')
results['SVM (RBF)'] = f1

### 5.5 XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', num_class=5,
    eval_metric='mlogloss', random_state=SEED,
    use_label_encoder=False
)
xgb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights_train)
f1, _ = evaluate_model(xgb_model, X_val_scaled, y_val, 'XGBoost')
results['XGBoost'] = f1

### 5.6 LightGBM

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, num_leaves=63,
    class_weight='balanced', random_state=SEED, verbose=-1
)
lgb_model.fit(X_train_scaled, y_train)
f1, _ = evaluate_model(lgb_model, X_val_scaled, y_val, 'LightGBM')
results['LightGBM'] = f1

### 5.7 MLP Neural Network

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128), activation='relu',
    max_iter=500, early_stopping=True, validation_fraction=0.1,
    batch_size=64, learning_rate='adaptive', learning_rate_init=0.001,
    random_state=SEED
)
mlp.fit(X_train_scaled, y_train)
f1, _ = evaluate_model(mlp, X_val_scaled, y_val, 'MLP Neural Network')
results['MLP'] = f1

### 5.8 Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=SEED
)
gb.fit(X_train_scaled, y_train, sample_weight=sample_weights_train)
f1, _ = evaluate_model(gb, X_val_scaled, y_val, 'Gradient Boosting')
results['Gradient Boosting'] = f1

## 6. Model Comparison

In [ ]:
print('\n' + '='*60)
print('MODEL COMPARISON (Validation Macro F1)')
print('='*60)
for name, f1 in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(f1 * 50)
    print(f'{name:25s} | {f1:.4f} | {bar}')

best_name = max(results, key=results.get)
print(f'\nBest model: {best_name} (F1={results[best_name]:.4f})')

## 7. Ensemble: Stacking Classifier

Combine the top models using stacking for potentially better performance.

In [ ]:
# Soft Voting Ensemble
voting_clf = VotingClassifier(
    estimators=[
        ('svm', SVC(C=10, kernel='rbf', gamma='scale', class_weight='balanced', 
                     random_state=SEED, probability=True)),
        ('xgb', xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=SEED, use_label_encoder=False, eval_metric='mlogloss')),
        ('lgb', lgb.LGBMClassifier(n_estimators=500, max_depth=8, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8, num_leaves=63,
                                    class_weight='balanced', random_state=SEED, verbose=-1)),
        ('lr', LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced',
                                  multi_class='multinomial', random_state=SEED)),
    ],
    voting='soft'
)
voting_clf.fit(X_train_scaled, y_train)
f1, _ = evaluate_model(voting_clf, X_val_scaled, y_val, 'Soft Voting Ensemble')
results['Soft Voting'] = f1

In [ ]:
# Stacking Ensemble
stacking_clf = StackingClassifier(
    estimators=[
        ('svm', SVC(C=10, kernel='rbf', gamma='scale', class_weight='balanced', 
                     random_state=SEED, probability=True)),
        ('xgb', xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=SEED, use_label_encoder=False, eval_metric='mlogloss')),
        ('lgb', lgb.LGBMClassifier(n_estimators=500, max_depth=8, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8, num_leaves=63,
                                    class_weight='balanced', random_state=SEED, verbose=-1)),
        ('rf', RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_leaf=2,
                                      class_weight='balanced', random_state=SEED)),
    ],
    final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED),
    cv=5, passthrough=False
)
stacking_clf.fit(X_train_scaled, y_train)
f1, _ = evaluate_model(stacking_clf, X_val_scaled, y_val, 'Stacking Ensemble')
results['Stacking'] = f1

## 8. Final Model Comparison & Select Best

In [ ]:
print('\n' + '='*60)
print('FINAL MODEL COMPARISON (Validation Macro F1)')
print('='*60)
for name, f1 in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(f1 * 50)
    print(f'{name:25s} | {f1:.4f} | {bar}')

best_name = max(results, key=results.get)
print(f'\n*** Best model: {best_name} (Macro F1={results[best_name]:.4f}) ***')

## 9. Generate Kaggle Submission

In [ ]:
# Select best model for final prediction
model_map = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'SVM (RBF)': svm,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'MLP': mlp,
    'Gradient Boosting': gb,
    'Soft Voting': voting_clf,
    'Stacking': stacking_clf,
}

best_model = model_map[best_name]
print(f'Using best model: {best_name}')

# Predict on test set
y_test_pred = best_model.predict(X_test_scaled)
y_test_labels = label_encoder.inverse_transform(y_test_pred)

# Create submission DataFrame
submission = pd.DataFrame({
    'case_id': [int(cid) for cid in test_ids],
    'Overall_class': y_test_labels
})

# Sort by case_id to match sample_submission.csv format
sample_sub = pd.read_csv(os.path.join(BASE_DIR, 'sample_submission.csv'))
submission = submission.set_index('case_id').loc[sample_sub['case_id']].reset_index()

# Save
submission.to_csv('submission.csv', index=False)
print(f'\nSubmission saved: submission.csv')
print(f'Shape: {submission.shape}')
print(f'\nPrediction distribution:')
print(submission['Overall_class'].value_counts())
print(f'\nFirst 10 rows:')
print(submission.head(10))

## 10. Modality Ablation Study

Analyze the contribution of each modality to classification performance.

In [ ]:
# Test each feature group individually and in combinations
feature_groups = {
    'Image Only': (X_img_train_pca, X_img_val_pca),
    'Radiomics Only': (X_rad_train, X_rad_val),
    'Clinical Only': (X_clin_train, X_clin_val),
    'Text Only': (X_text_train, X_text_val),
    'Image+Radiomics': (np.hstack([X_img_train_pca, X_rad_train]), np.hstack([X_img_val_pca, X_rad_val])),
    'Image+Clinical': (np.hstack([X_img_train_pca, X_clin_train]), np.hstack([X_img_val_pca, X_clin_val])),
    'Image+Text': (np.hstack([X_img_train_pca, X_text_train]), np.hstack([X_img_val_pca, X_text_val])),
    'All Modalities': (X_train_all, X_val_all),
}

ablation_results = {}
for name, (X_tr, X_va) in feature_groups.items():
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    
    model = lgb.LGBMClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        class_weight='balanced', random_state=SEED, verbose=-1
    )
    model.fit(X_tr_s, y_train)
    y_pred = model.predict(X_va_s)
    f1 = f1_score(y_val, y_pred, average='macro')
    ablation_results[name] = f1

print('\n' + '='*60)
print('MODALITY ABLATION STUDY (LightGBM, Macro F1)')
print('='*60)
for name, f1 in sorted(ablation_results.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(f1 * 50)
    print(f'{name:25s} | {f1:.4f} | {bar}')

## 11. Verification

In [ ]:
# Verify submission format
sample = pd.read_csv('sample_submission.csv')
sub = pd.read_csv('submission.csv')

print('Format verification:')
print(f'  Column names match: {list(sub.columns) == list(sample.columns)}')
print(f'  Row count match: {len(sub)} == {len(sample)} -> {len(sub) == len(sample)}')
print(f'  case_id match: {(sub["case_id"].values == sample["case_id"].values).all()}')
print(f'  No NaN in predictions: {sub["Overall_class"].isna().sum() == 0}')

valid_classes = ['Brain Metastase Tumour', 'Glioma', 'Meningioma', 
                 'Pineal tumour and Choroid plexus tumour', 'Tumors of the sellar region']
all_valid = sub['Overall_class'].isin(valid_classes).all()
print(f'  All classes valid: {all_valid}')
print(f'\nSubmission is READY for Kaggle upload!' if all_valid else '\nWARNING: Invalid class names found!')